### 1. Import Library
Memuat library NumPy sebagai fondasi komputasi numerik. NumPy merupakan tulang punggung hampir seluruh ekosistem ML/DL (PyTorch tensor, embedding matrix, dll.) dibangun di atasnya.

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")

NumPy version: 2.2.6


### 2. Membuat Array (`np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`)
Array NumPy adalah struktur data utama. Mengetahui cara membuatnya penting karena input dan output model (embedding, logit, label) berbentuk NumPy array atau padanannya di PyTorch.

In [2]:
# Dari list Python biasa
a = np.array([1, 2, 3, 4, 5])
print("Array dari list:", a)

# Array 2D (seperti batch embedding)
b = np.array([[1.0, 0.5, 0.2],
              [0.8, 0.3, 0.9]])
print("Array 2D:\n", b)

# Array nol dan satu — init bobot model
zeros = np.zeros((3, 4))
ones  = np.ones((2, 3))
print("Zeros:\n", zeros)
print("Ones:\n", ones)

# Urutan angka — epoch / step index
steps = np.arange(0, 10, 2)
print("Arange:", steps)

# Spasi merata — learning rate scheduler
lr_schedule = np.linspace(1e-4, 1e-6, num=5)
print("Linspace (LR schedule):", lr_schedule)

Array dari list: [1 2 3 4 5]
Array 2D:
 [[1.  0.5 0.2]
 [0.8 0.3 0.9]]
Zeros:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Ones:
 [[1. 1. 1.]
 [1. 1. 1.]]
Arange: [0 2 4 6 8]
Linspace (LR schedule): [1.000e-04 7.525e-05 5.050e-05 2.575e-05 1.000e-06]


### 3. Shape, Dtype, dan Reshape
Atribut `shape` dan `dtype` adalah hal *pertama* yang dicek saat debugging model. Kesalahan dimensi tensor adalah sumber error paling umum di PyTorch/TensorFlow. `reshape` dan `squeeze`/`unsqueeze` digunakan untuk menyesuaikan dimensi batch.

In [3]:
# Simulasi batch embedding: 4 sampel, tiap sampel 8 dimensi
embedding = np.random.randn(4, 8)

print("Shape:", embedding.shape)       # (4, 8)
print("Dtype:", embedding.dtype)       # float64
print("Ndim:", embedding.ndim)         # 2
print("Size (total elemen):", embedding.size)  # 32

# Reshape: flatten ke 1D
flat = embedding.reshape(-1)
print("Flatten shape:", flat.shape)    # (32,)

# Reshape ke (2, 16)
reshaped = embedding.reshape(2, 16)
print("Reshaped (2,16):", reshaped.shape)

# Cast dtype — PyTorch biasanya butuh float32
embedding_f32 = embedding.astype(np.float32)
print("Setelah astype float32:", embedding_f32.dtype)

Shape: (4, 8)
Dtype: float64
Ndim: 2
Size (total elemen): 32
Flatten shape: (32,)
Reshaped (2,16): (2, 16)
Setelah astype float32: float32


### 4. Operasi Matematika Dasar dan Broadcasting
NumPy melakukan operasi elemen-per-elemen secara otomatis, dan **broadcasting** memungkinkan operasi antar array yang berbeda dimensi tanpa loop Python. Ini adalah dasar dari mekanisme attention score, normalisasi, dan banyak kalkulasi model lainnya.

In [4]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([0.5, 1.5, 0.5, 1.0])

print("Penjumlahan:", a + b)
print("Perkalian elemen:", a * b)
print("Pangkat 2:", a ** 2)
print("Akar kuadrat:", np.sqrt(a))
print("Exp (softmax dasar):", np.exp(a))

# Broadcasting: tambahkan bias ke setiap baris embedding
embedding = np.ones((3, 4))  # 3 sampel, 4 dim
bias = np.array([0.1, 0.2, 0.3, 0.4])  # 1 bias per dimensi

result = embedding + bias  # broadcast: (3,4) + (4,) → (3,4)
print("\nEmbedding + bias (broadcasting):\n", result)

Penjumlahan: [1.5 3.5 3.5 5. ]
Perkalian elemen: [0.5 3.  1.5 4. ]
Pangkat 2: [ 1.  4.  9. 16.]
Akar kuadrat: [1.         1.41421356 1.73205081 2.        ]
Exp (softmax dasar): [ 2.71828183  7.3890561  20.08553692 54.59815003]

Embedding + bias (broadcasting):
 [[1.1 1.2 1.3 1.4]
 [1.1 1.2 1.3 1.4]
 [1.1 1.2 1.3 1.4]]


### 5. Statistik Deskriptif (`mean`, `std`, `min`, `max`, `sum`)
Fungsi statistik ini krusial untuk normalisasi data, memantau loss selama training, serta analisis distribusi token panjang (seperti yang dilakukan sebelum menentukan `MAX_SEQ_LEN`).

In [5]:
# Simulasi loss per batch selama training
losses = np.array([2.51, 2.38, 2.10, 1.95, 1.82, 1.71, 1.63, 1.58, 1.51, 1.47])

print("Mean loss:",   np.mean(losses))
print("Std loss:",    np.std(losses))
print("Min loss:",    np.min(losses))
print("Max loss:",    np.max(losses))
print("Sum loss:",    np.sum(losses))

# Axis pada array 2D — misal matrix logit (batch=4, kelas=3)
logits = np.array([[2.1, 0.5, -1.0],
                   [0.3, 3.2,  0.8],
                   [-0.5, 1.1, 2.9],
                   [1.0,  0.0, 0.7]])

print("\nMean per kelas (axis=0):", np.mean(logits, axis=0))  # rata2 tiap kolom/kelas
print("Max per sampel (axis=1):",  np.max(logits, axis=1))   # max score per baris/sampel
print("Argmax per sampel:",        np.argmax(logits, axis=1))  # prediksi kelas

Mean loss: 1.866
Std loss: 0.34470857256529025
Min loss: 1.47
Max loss: 2.51
Sum loss: 18.66

Mean per kelas (axis=0): [0.725 1.2   0.85 ]
Max per sampel (axis=1): [2.1 3.2 2.9 1. ]
Argmax per sampel: [0 1 2 0]


### 6. Indexing, Slicing, dan Boolean Masking
Teknik mengakses subset array dengan cepat. Boolean masking sangat sering digunakan untuk memfilter token padding (`attention_mask`), memilih sampel bertarget tertentu, atau mengekstrak prediksi yang benar dari sekumpulan hasil inferensi.

In [6]:
tokens = np.array([101, 2023, 2003, 1037, 3231, 102, 0, 0, 0, 0])  # 0 = padding

# Slicing biasa
print("Token pertama:", tokens[0])
print("3 token pertama:", tokens[:3])
print("Token terakhir:", tokens[-1])

# Boolean masking: ambil hanya token yang bukan padding
attention_mask = tokens != 0
print("Attention mask:", attention_mask.astype(int))
print("Token valid:", tokens[attention_mask])

# Slicing 2D — ambil baris/kolom tertentu dari batch embedding
embedding_batch = np.random.randn(5, 8)  # 5 sampel, 8 dim
print("\nBaris ke-2 (sampel ke-3):", embedding_batch[2])
print("Kolom 0-3 dari 2 sampel pertama:\n", embedding_batch[:2, :4])

# Fancy indexing: pilih sampel indeks [0, 2, 4]
selected = embedding_batch[[0, 2, 4]]
print("Sampel yang dipilih (fancy indexing) shape:", selected.shape)

Token pertama: 101
3 token pertama: [ 101 2023 2003]
Token terakhir: 0
Attention mask: [1 1 1 1 1 1 0 0 0 0]
Token valid: [ 101 2023 2003 1037 3231  102]

Baris ke-2 (sampel ke-3): [-1.69295805e+00  9.69220214e-02  8.38823674e-01 -8.21184686e-01
  1.27013129e+00  3.76543417e-01  1.64178711e-03 -4.62266872e-02]
Kolom 0-3 dari 2 sampel pertama:
 [[-0.5235104   0.03153434  0.68727577 -0.99240829]
 [-0.87867535 -0.45853928 -1.77372602  0.18218598]]
Sampel yang dipilih (fancy indexing) shape: (3, 8)


### 7. Operasi Matrix (`dot`, `matmul`, `transpose`)
Perkalian matrix adalah inti dari semua lapisan neural network: `y = xW + b`. Selain itu, *dot product* antara vektor embedding digunakan untuk menghitung cosine similarity dan attention score.

In [7]:
# Simulasi: x = input (batch_size=3, input_dim=4)
#           W = bobot linear layer (input_dim=4, output_dim=2)
x = np.random.randn(3, 4)
W = np.random.randn(4, 2)
b = np.array([0.1, -0.1])

# Forward pass linear layer
y = x @ W + b       # atau: np.matmul(x, W) + b
print("Output linear layer (3,2):\n", y)

# Transpose
print("\nShape W:", W.shape)
print("Shape W.T:", W.T.shape)

# Dot product dua vektor (cosine similarity sederhana)
v1 = np.array([1.0, 0.5, -0.3])
v2 = np.array([0.8, 0.2,  0.1])

dot_product  = np.dot(v1, v2)
cosine_sim   = dot_product / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(f"\nDot product: {dot_product:.4f}")
print(f"Cosine similarity: {cosine_sim:.4f}")

Output linear layer (3,2):
 [[ 0.82861757 -0.0973426 ]
 [-1.02353224 -0.55806313]
 [ 0.55939809  0.46038694]]

Shape W: (4, 2)
Shape W.T: (2, 4)

Dot product: 0.8700
Cosine similarity: 0.9048


### 8. Fungsi Aktivasi Manual (Softmax, Sigmoid, ReLU)
Memahami cara mengimplementasikan fungsi aktivasi secara manual dari operasi NumPy dasar adalah kunci untuk memahami alur komputasi model dari bawah ke atas, termasuk cara kerja numerik PyTorch.

In [8]:
logits = np.array([2.1, 0.5, -1.0, 3.2])  # raw output model

# Softmax: konversi logit ke probabilitas (jumlah = 1)
def softmax(x):
    e_x = np.exp(x - np.max(x))  # numerical stability trick
    return e_x / e_x.sum()

probs = softmax(logits)
print("Logits:", logits)
print("Softmax (probabilitas):", probs)
print("Sum probabilitas:", probs.sum())  # harus = 1.0

# Sigmoid: output biner [0, 1]
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print("\nSigmoid:", sigmoid(logits))

# ReLU: max(0, x)
def relu(x):
    return np.maximum(0, x)

print("ReLU:", relu(logits))

# Prediksi kelas dari probabilitas
predicted_class = np.argmax(probs)
print(f"\nKelas prediksi: {predicted_class} (probabilitas: {probs[predicted_class]:.4f})")

Logits: [ 2.1  0.5 -1.   3.2]
Softmax (probabilitas): [0.23523258 0.04749264 0.01059704 0.70667774]
Sum probabilitas: 0.9999999999999999

Sigmoid: [0.89090318 0.62245933 0.26894142 0.96083428]
ReLU: [2.1 0.5 0.  3.2]

Kelas prediksi: 3 (probabilitas: 0.7067)


### 9. Random Number Generation (`np.random`)
Pembangkitan bilangan acak digunakan untuk inisialisasi bobot model (weight initialization), shuffling dataset, simulasi data, dan reproducibility lewat `seed`. Tanpa seed tetap, hasil training tidak bisa direproduksi.

In [9]:
# Set seed untuk reproducibility
np.random.seed(42)

# Distribusi normal (Gaussian) — standar untuk init bobot
weights = np.random.randn(4, 4)
print("Weight init (normal):\n", weights.round(3))

# Uniform [0, 1) — inisialisasi Glorot/Xavier sederhana
uniform = np.random.uniform(low=-0.5, high=0.5, size=(3, 3))
print("\nUniform init:\n", uniform.round(3))

# Random integer — simulasi label kelas
fake_labels = np.random.randint(0, 3, size=10)
print("\nFake labels:", fake_labels)

# Shuffling indeks — cara numpy untuk shuffle dataset sebelum batching
indices = np.arange(10)
np.random.shuffle(indices)
print("Shuffled indices:", indices)

# Choice tanpa pengembalian — random sampling
sampled = np.random.choice(100, size=10, replace=False)
print("Random sample 10 dari 100:", sampled)

Weight init (normal):
 [[ 0.497 -0.138  0.648  1.523]
 [-0.234 -0.234  1.579  0.767]
 [-0.469  0.543 -0.463 -0.466]
 [ 0.242 -1.913 -1.725 -0.562]]

Uniform init:
 [[ 0.112 -0.361 -0.208]
 [-0.134 -0.044  0.285]
 [-0.3    0.014  0.092]]

Fake labels: [2 0 2 2 0 0 2 1 0 1]
Shuffled indices: [0 5 2 6 3 7 4 9 8 1]
Random sample 10 dari 100: [15 94 92 89 88 65 18 45 36 55]


### 10. Stacking dan Concatenation (`np.stack`, `np.concatenate`, `np.vstack`, `np.hstack`)
Saat membangun batch dari sekumpulan sampel individual, kita perlu menggabungkan array. `np.stack` membuat dimensi baru (seperti menumpuk embedding jadi batch), sedangkan `np.concatenate` menggabungkan sepanjang dimensi yang sudah ada.

In [10]:
# Simulasi: tiap sampel menghasilkan embedding 1D
emb1 = np.array([0.1, 0.5, 0.9])
emb2 = np.array([0.3, 0.7, 0.2])
emb3 = np.array([0.8, 0.1, 0.4])

# stack: buat dimensi batch baru → (3, 3)
batch = np.stack([emb1, emb2, emb3], axis=0)
print("Batch (stack, axis=0):\n", batch, "shape:", batch.shape)

# concatenate: gabung sepanjang axis yang ada → (9,)
concat = np.concatenate([emb1, emb2, emb3])
print("\nConcatenate:", concat, "shape:", concat.shape)

# vstack (vertical): tumpuk baris → (6, 3)
a = np.ones((3, 3))
b = np.zeros((3, 3))
vertical = np.vstack([a, b])
print("\nvstack shape:", vertical.shape)

# hstack (horizontal): tempel kolom → (3, 6)
horizontal = np.hstack([a, b])
print("hstack shape:", horizontal.shape)

Batch (stack, axis=0):
 [[0.1 0.5 0.9]
 [0.3 0.7 0.2]
 [0.8 0.1 0.4]] shape: (3, 3)

Concatenate: [0.1 0.5 0.9 0.3 0.7 0.2 0.8 0.1 0.4] shape: (9,)

vstack shape: (6, 3)
hstack shape: (3, 6)


### 11. Normalisasi Data (Min-Max dan Z-Score)
Normalisasi fitur numerik sangat penting sebelum memasukkannya ke model agar gradient tidak meledak atau menghilang. Z-score (standardisasi) adalah teknik paling umum di deep learning, sementara min-max scaling berguna ketika rentang nilai diketahui.

In [11]:
# Data panjang kalimat dari dataset (dalam jumlah token)
token_lengths = np.array([15, 23, 8, 45, 31, 12, 67, 19, 52, 28], dtype=np.float64)

# Min-Max Normalization → rentang [0, 1]
def minmax_normalize(x):
    return (x - x.min()) / (x.max() - x.min())

normalized_minmax = minmax_normalize(token_lengths)
print("Original lengths:", token_lengths)
print("Min-Max normalized:", normalized_minmax.round(3))
print(f"  Range setelah normalisasi: [{normalized_minmax.min():.3f}, {normalized_minmax.max():.3f}]")

# Z-Score Standardization → mean=0, std=1
def zscore_normalize(x):
    return (x - x.mean()) / x.std()

normalized_zscore = zscore_normalize(token_lengths)
print("\nZ-Score normalized:", normalized_zscore.round(3))
print(f"  Mean: {normalized_zscore.mean():.6f} (mendekati 0)")
print(f"  Std:  {normalized_zscore.std():.6f} (mendekati 1)")

Original lengths: [15. 23.  8. 45. 31. 12. 67. 19. 52. 28.]
Min-Max normalized: [0.119 0.254 0.    0.627 0.39  0.068 1.    0.186 0.746 0.339]
  Range setelah normalisasi: [0.000, 1.000]

Z-Score normalized: [-0.827 -0.386 -1.214  0.827  0.055 -0.993  2.041 -0.607  1.214 -0.11 ]
  Mean: -0.000000 (mendekati 0)
  Std:  1.000000 (mendekati 1)


### 12. One-Hot Encoding Manual
One-hot encoding mengubah label kelas integer menjadi vektor biner. Ini adalah representasi dasar label untuk klasifikasi sebelum diproses oleh loss function seperti `CrossEntropyLoss` (yang menerima class index) atau `BCELoss` (yang menerima probabilitas).

In [12]:
# Label kelas integer untuk 6 sampel, 3 kelas
labels = np.array([0, 2, 1, 0, 2, 1])
num_classes = 3

# Cara NumPy untuk one-hot encoding
one_hot = np.eye(num_classes)[labels]
print("Labels integer:", labels)
print("One-Hot Encoded:\n", one_hot)

# Kebalikan: dari one-hot ke label integer menggunakan argmax
labels_restored = np.argmax(one_hot, axis=1)
print("\nLabel dari argmax one-hot:", labels_restored)
print("Sama dengan aslinya:", np.array_equal(labels, labels_restored))

# Simulasi hitung akurasi secara manual
y_true = np.array([0, 2, 1, 0, 2, 1])
y_pred = np.array([0, 1, 1, 0, 2, 0])  # 4 dari 6 benar

accuracy = np.mean(y_true == y_pred)
print(f"\nAkurasi manual: {accuracy:.4f} ({int(accuracy*len(y_true))}/{len(y_true)} benar)")

Labels integer: [0 2 1 0 2 1]
One-Hot Encoded:
 [[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]

Label dari argmax one-hot: [0 2 1 0 2 1]
Sama dengan aslinya: True

Akurasi manual: 0.6667 (4/6 benar)


### 13. Sorting dan Argsort
`argsort` mengembalikan *indeks* yang mengurutkan array — sangat berguna untuk ranking prediksi (top-k accuracy), mencari nearest neighbor dalam embedding space, dan menyusun hasil evaluasi.

In [13]:
# Probabilitas output model untuk 5 kelas
probs = np.array([0.05, 0.15, 0.60, 0.12, 0.08])
class_names = ["negatif", "netral", "positif", "campuran", "tidak_relevan"]

# Urutan ascending
sorted_indices = np.argsort(probs)            # index dari kecil ke besar
sorted_desc    = np.argsort(probs)[::-1]      # index dari besar ke kecil (flip)

print("Probabilitas:", probs)
print("Indeks terurut (asc):", sorted_indices)
print("Indeks terurut (desc):", sorted_desc)

# Top-3 prediksi
top3_idx = sorted_desc[:3]
print("\nTop-3 prediksi:")
for rank, idx in enumerate(top3_idx, 1):
    print(f"  Rank {rank}: {class_names[idx]} ({probs[idx]:.4f})")

# Sort array 1D biasa
losses_per_epoch = np.array([2.5, 1.8, 1.3, 1.7, 1.1])
print("\nLoss terurut (asc):", np.sort(losses_per_epoch))
print("Epoch terbaik (loss terendah): Epoch", np.argmin(losses_per_epoch) + 1)

Probabilitas: [0.05 0.15 0.6  0.12 0.08]
Indeks terurut (asc): [0 4 3 1 2]
Indeks terurut (desc): [2 1 3 4 0]

Top-3 prediksi:
  Rank 1: positif (0.6000)
  Rank 2: netral (0.1500)
  Rank 3: campuran (0.1200)

Loss terurut (asc): [1.1 1.3 1.7 1.8 2.5]
Epoch terbaik (loss terendah): Epoch 5


### 14. Save dan Load Array (`np.save`, `np.load`, `np.savetxt`)
Menyimpan embedding hasil model ke disk dan memuatnya kembali tanpa perlu menjalankan ulang inference adalah praktik yang wajib dikuasai saat bekerja dengan dataset besar atau embeddings yang mahal secara komputasi.

In [14]:
import os

# Simulasi embedding hasil encoder model
np.random.seed(0)
embeddings = np.random.randn(100, 768)  # 100 kalimat, dim=768 (BERT-base)
labels_arr = np.array([0, 1] * 50)     # 100 label biner

os.makedirs('output', exist_ok=True)

# Simpan satu array ke file .npy
np.save('output/embeddings.npy', embeddings)
print(f"Embeddings disimpan: shape {embeddings.shape}, dtype {embeddings.dtype}")

# Simpan beberapa array sekaligus ke file .npz (terkompresi)
np.savez('output/dataset.npz', embeddings=embeddings, labels=labels_arr)
print("Dataset (.npz) disimpan.")

# Load kembali
loaded_emb = np.load('output/embeddings.npy')
print(f"\nLoaded embeddings shape: {loaded_emb.shape}")

loaded_npz = np.load('output/dataset.npz')
print("Keys dalam .npz:", list(loaded_npz.keys()))
print("Labels dari .npz:", loaded_npz['labels'][:10])

# Simpan ke format teks (CSV-like) untuk inspeksi manual
small_emb = embeddings[:5, :4]  # ambil subset kecil
np.savetxt('output/small_embeddings.txt', small_emb, fmt='%.4f', delimiter=',')
print("\nSmall embeddings disimpan ke .txt")

Embeddings disimpan: shape (100, 768), dtype float64
Dataset (.npz) disimpan.

Loaded embeddings shape: (100, 768)
Keys dalam .npz: ['embeddings', 'labels']
Labels dari .npz: [0 1 0 1 0 1 0 1 0 1]

Small embeddings disimpan ke .txt
